In [1]:
%reset -f

In [7]:
%cd anomalib

/workspaces/eng-ai-agents/assignments/assignment-2/anomalib


/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
!pip install .
!pip install qdrant-client

Defaulting to user installation because normal site-packages is not writeable
ERROR: Directory '.' is not installable. Neither 'setup.py' nor 'pyproject.toml' found.

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


# Importing Packages

In [81]:
from typing import Any

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from PIL import Image
import torch
from torchvision.transforms import Normalize
from torchvision.transforms import ToPILImage
from torchmetrics import AUROC

from anomalib.data import MVTec
from anomalib.data.utils import read_image
from anomalib.deploy import ExportType, OpenVINOInferencer
from anomalib.engine import Engine
from anomalib.models import Padim,Patchcore,EfficientAd
from anomalib.utils.visualization import ImageResult


In [ ]:
datamodule = MVTec(num_workers=0)
datamodule.prepare_data()  # Downloads the dataset if it's not in the specified `root` directory
datamodule.setup()  # Create train/val/test/prediction sets.

# Get a batch of data from the validation DataLoader
i, data = next(enumerate(datamodule.val_dataloader()))

print(type(data))

#Checks for data structure
if isinstance(data, dict):
    for key, value in data.items():
        print(f"{key}: {type(value)}, {value.shape if hasattr(value, 'shape') else 'N/A'}")
elif isinstance(data, (list, tuple)):
    for idx, value in enumerate(data):
        print(f"Element {idx}: {type(value)}, {value.shape if hasattr(value, 'shape') else 'N/A'}")


INFO:anomalib.data.image.mvtec:Found the dataset.


<class 'dict'>
image_path: <class 'list'>, N/A
label: <class 'torch.Tensor'>, torch.Size([32])
image: <class 'torch.Tensor'>, torch.Size([32, 3, 900, 900])
mask: <class 'torch.Tensor'>, torch.Size([32, 900, 900])


# Initializing models and engine

In [76]:
patchCore = Patchcore(pre_trained=True)
efficientAd = EfficientAd()
engine = Engine()

INFO:anomalib.models.components.base.anomaly_module:Initializing Patchcore model.
INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/wide_resnet50_2.racm_in1k)
INFO:timm.models._hub:[timm/wide_resnet50_2.racm_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:timm.models._builder:Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO:anomalib.models.components.base.anomaly_module:Initializing EfficientAd model.


# Data processing

In [97]:
#Categories mentioned in assignment
categories = ['tile','leather','grid']
datamodules = {cat: MVTec(category=cat,train_batch_size=1) for cat in categories}
# Prepare and setup dataset
for cat, datamodule in datamodules.items():
    datamodule.prepare_data()
    datamodule.setup()

# Check dataset shape for one sample
i, data = next(enumerate(datamodules["tile"].val_dataloader()))

print(f"Sample Shape - Image: {data['image'].shape}, Mask: {data['mask'].shape}")




INFO:anomalib.data.image.mvtec:Found the dataset.


INFO:anomalib.data.image.mvtec:Found the dataset.
INFO:anomalib.data.image.mvtec:Found the dataset.


Sample Shape - Image: torch.Size([32, 3, 840, 840]), Mask: torch.Size([32, 840, 840])


In [93]:
aurocResults = {}
for model_name, model in [("PatchCore", patchCore), ("EfficientAD", efficientAd)]:
    aurocResults[model_name] = {}

    for cat, datamodule in datamodules.items():
        print(f"Training {model_name} for {cat}")
        engine.fit(model=model, datamodule=datamodule)

        print(f"Evaluating {model_name} for {cat}")
        test_results = engine.test(
            model=model,
            datamodule=datamodule,
            ckpt_path=engine.trainer.checkpoint_callback.best_model_path,
        )

        # Extract AUROC Score
        if "AUROC" in test_results:
            aurocScore = test_results[0]["image_AUROC"]
            aurocResults[model_name][cat] = aurocScore
            print(f"{model_name} AUROC for {cat}: {aurocScore:.4f}")
        else:
            print(f"Key 'AUROC' not found in test_results. Available keys: {test_results}")

INFO:anomalib.data.image.mvtec:Found the dataset.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training PatchCore for tile


/home/vscode/.local/lib/python3.11/site-packages/lightning/pytorch/core/optimizer.py:182: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer

  | Name                  | Type                     | Params | Mode 
---------------------------------------------------------------------------
0 | model                 | PatchcoreModel           | 24.9 M | train
1 | _transform            | Compose                  | 0      | train
2 | normalization_metrics | MetricCollection         | 0      | train
3 | image_threshold       | F1AdaptiveThreshold      | 0      | train
4 | pixel_threshold       | F1AdaptiveThreshold      | 0      | train
5 | image_metrics         | AnomalibMetricCollection | 0      | train
6 | pixel_metrics         | AnomalibMetricCollection | 0      | train
---------------------------------------------------------------------------
24.9 M    Trainable params
0         Non-trainable params
24.9 M    Total params
99.450    Total estimate

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.69 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 8.98 GiB is allocated by PyTorch, and 1.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Patch Core

In [ ]:
aurocResults["PatchCore"] = {}
for cat, datamodule in datamodules.items():
    print(f"Training PatchCore for {cat}")
    engine.fit(model=patchCore, datamodule=datamodule)

    print(f"Evaluating PatchCore for {cat}")
    test_results = engine.test(
        model=patchCore,
        datamodule=datamodule,
        ckpt_path=engine.trainer.checkpoint_callback.best_model_path,
    )

    aurocResults["PatchCore"][cat] = test_results[0]["image_AUROC"]

INFO:anomalib.data.image.mvtec:Found the dataset.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training PatchCore for tile



  | Name                               | Type                     | Params | Mode 
----------------------------------------------------------------------------------------
0 | model                              | PatchcoreModel           | 24.9 M | train
1 | _transform                         | Compose                  | 0      | train
2 | normalization_metrics              | MetricCollection         | 0      | train
3 | image_threshold                    | F1AdaptiveThreshold      | 0      | train
4 | pixel_threshold                    | F1AdaptiveThreshold      | 0      | train
5 | image_metrics                      | AnomalibMetricCollection | 0      | train
6 | pixel_metrics                      | AnomalibMetricCollection | 0      | train
7 | box_scores_normalization_metrics   | MinMax                   | 0      | train
8 | anomaly_maps_normalization_metrics | MinMax                   | 0      | train
9 | pred_scores_normalization_metrics  | MinMax                   | 0      | tra

Evaluating PatchCore for tile


...

Evaluating PatchCore for grid


INFO:anomalib.data.image.mvtec:Found the dataset.
Restoring states from the checkpoint path at /workspaces/eng-ai-agents/assignments/assignment-2/results/Patchcore/MVTec/tile/v0/weights/lightning/model.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /workspaces/eng-ai-agents/assignments/assignment-2/results/Patchcore/MVTec/tile/v0/weights/lightning/model.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

INFO:anomalib.callbacks.timer:Testing took 22.68462300300598 seconds
Throughput (batch_size=32) : 3.4384525583547974 FPS


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            0.5            │
│       image_F1Score       │    0.8444444537162781     │
│        pixel_AUROC        │    0.2669127285480499     │
│       pixel_F1Score       │   0.016495464369654655    │
└───────────────────────────┴───────────────────────────┘

## EfficientAD

In [80]:
print(dir(datamodule))

['CHECKPOINT_HYPER_PARAMS_KEY', 'CHECKPOINT_HYPER_PARAMS_NAME', 'CHECKPOINT_HYPER_PARAMS_TYPE', '__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__jit_unused_properties__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_category', '_create_test_split', '_create_val_split', '_eval_transform', '_hparams', '_is_setup', '_log_hyperparams', '_samples', '_set_hparams', '_setup', '_to_hparams_dict', '_train_transform', 'allow_zero_length_dataloader_with_multiple_devices', 'category', 'collate_fn', 'eval_batch_size', 'eval_transform', 'from_config', 'from_datasets', 'hparams', 'hparams_initial', 'image_size', 'load_from_checkpoint', 'load_state_dict', 'name', 'num_

In [85]:
for cat, datamodule in datamodules.items():
    sample_batch = next(iter(datamodule.train_dataloader()))
    print(f"{cat} - Image shape: {sample_batch['image'].shape}")


tile - Image shape: torch.Size([1, 3, 224, 224])
leather - Image shape: torch.Size([32, 3, 224, 224])
grid - Image shape: torch.Size([32, 3, 224, 224])


In [87]:
print(efficientAd.hparams)  # Check model hyperparameters
print(datamodule.train_transform)



"imagenet_dir":         ./datasets/imagenette
"lr":                   0.0001
"model_size":           EfficientAdModelSize.S
"pad_maps":             True
"padding":              False
"teacher_out_channels": 384
"weight_decay":         1e-05
Compose(
      Resize(size=[256, 256], interpolation=InterpolationMode.BILINEAR, antialias=True)
      CenterCrop(size=(224, 224))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)


In [98]:
ead_model = EfficientAd()
engine = Engine(max_epochs=1)
engine.fit(ead_model,datamodule=datamodule)
engine.predict(ead_model,datamodule=datamodule)
print(engine)


INFO:anomalib.models.components.base.anomaly_module:Initializing EfficientAd model.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
INFO:anomalib.data.image.mvtec:Found the dataset.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type                     | Params | Mode 
---------------------------------------------------------------------------
0 | model                 | EfficientAdModel         | 8.1 M  | train
1 | _transform            | Compose                  | 0      | train
2 | normalization_metrics | MetricCollection         | 0      | train
3 | image_threshold       | F1AdaptiveThreshold      | 0      | train
4 | pixel_threshold       | F1AdaptiveThreshold      | 0      | train
5 | image_metrics         | AnomalibMetricCollection | 0      | train
6 | pixel_metrics         | AnomalibMetricCollection | 0      | train
-----------------------------------------------------------------------

Training: |          | 0/? [00:00<?, ?it/s]

INFO:anomalib.models.image.efficient_ad.lightning_model:Load pretrained teacher model from pre_trained/efficientad_pretrained_weights/pretrained_teacher_small.pth
/home/vscode/.local/lib/python3.11/site-packages/anomalib/models/image/efficient_ad/lightning_model.py:98: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you do

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:anomalib.models.image.efficient_ad.lightning_model:Calculate Validation Dataset Quantiles
Calculate Validation Dataset Quantiles: 100%|██████████| 3/3 [00:07<00:00,  2.35s/it]
`Trainer.fit` stopped: `max_epochs=1` reached.
INFO:anomalib.callbacks.timer:Training took 766.53 seconds
INFO:anomalib.data.image.mvtec:Found the dataset.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]